# 03_create_ko_en_dictionary

Workflow:
1. `data/raw` 하위 JSON 파일 로드
2. 토크나이징
3. 단어 사전 생성 (TF/IDF 기반)
4. 단어 사전 저장


In [1]:
from __future__ import annotations

import json
import math
import re
from collections import Counter
from pathlib import Path

import pandas as pd

PROJECT_ROOT = Path.cwd().parent.resolve()
RAW_DIR = PROJECT_ROOT / 'data' / 'raw'
OUT_DIR = PROJECT_ROOT / 'data' / 'processed'
OUT_DIR.mkdir(parents=True, exist_ok=True)

KOREAN_STOPWORDS_PATH = RAW_DIR / 'korean_stopwords.json'
ENGLISH_STOPWORDS_PATH = RAW_DIR / 'english_stopwords.json'
OUTPUT_TFIDF_CSV = OUT_DIR / 'keyword_tfidf_scores.csv'
OUTPUT_DICT_JSON = RAW_DIR / 'ko_en_dictionary_auto.json'

print('PROJECT_ROOT =', PROJECT_ROOT)
print('RAW_DIR =', RAW_DIR)
print('KOREAN_STOPWORDS_PATH =', KOREAN_STOPWORDS_PATH)
print('ENGLISH_STOPWORDS_PATH =', ENGLISH_STOPWORDS_PATH)
print('OUTPUT_TFIDF_CSV =', OUTPUT_TFIDF_CSV)
print('OUTPUT_DICT_JSON =', OUTPUT_DICT_JSON)


PROJECT_ROOT = D:\AI\projects\Medical-Chatbot
RAW_DIR = D:\AI\projects\Medical-Chatbot\data\raw
KOREAN_STOPWORDS_PATH = D:\AI\projects\Medical-Chatbot\data\raw\korean_stopwords.json
ENGLISH_STOPWORDS_PATH = D:\AI\projects\Medical-Chatbot\data\raw\english_stopwords.json
OUTPUT_TFIDF_CSV = D:\AI\projects\Medical-Chatbot\data\processed\keyword_tfidf_scores.csv
OUTPUT_DICT_JSON = D:\AI\projects\Medical-Chatbot\data\raw\ko_en_dictionary_auto.json


In [2]:
# 1) data/raw 하위의 json 파일 로드
TARGET_JSON_FILES = [
    "TL_내과_통합.json",
    "VL_내과_통합.json",
    "TS_국문_온라인 의료 정보 제공 사이트_merged.json",
    "TS_국문_의학 교과서_merged.json",
    "TS_국문_학회 가이드라인_merged.json",
]

json_files = [RAW_DIR / name for name in TARGET_JSON_FILES]
print(f"target json files: {len(json_files)}")
for p in json_files:
    print(" -", p.name, "(exists)" if p.exists() else "(missing)")


def iter_records(raw):
    if isinstance(raw, list):
        for row in raw:
            if isinstance(row, dict):
                yield row
    elif isinstance(raw, dict):
        for v in raw.values():
            if isinstance(v, list):
                for row in v:
                    if isinstance(row, dict):
                        yield row
            elif isinstance(v, dict):
                yield v


def extract_text(record: dict) -> str:
    fields = []
    for key in ("question", "answer", "content", "title"):
        value = record.get(key)
        if isinstance(value, str) and value.strip():
            fields.append(value.strip())
    return "\n".join(fields)


documents = []
for path in json_files:
    if not path.exists():
        print(f"[skip] missing file: {path.name}")
        continue

    try:
        raw = json.loads(path.read_text(encoding="utf-8"))
    except Exception as e:
        print(f"[skip] {path.name}: {e}")
        continue

    for rec in iter_records(raw):
        text = extract_text(rec)
        if text:
            documents.append(text)

print("loaded documents =", len(documents))
print("sample doc =")
print(documents[0][:500] if documents else "(empty)")


target json files: 5
 - TL_내과_통합.json (exists)
 - VL_내과_통합.json (exists)
 - TS_국문_온라인 의료 정보 제공 사이트_merged.json (exists)
 - TS_국문_의학 교과서_merged.json (exists)
 - TS_국문_학회 가이드라인_merged.json (exists)
loaded documents = 14994
sample doc =
23세 여자가 3개월 전부터 기침을 한다며 내원했다. 기침은 밤에 누워 자려고 할 때 심해진다고 한다. 1년 전에도 같은 시기에 기침이 3개월 동안 지속되다가 저절로 호전된 병력이 있다. 콧물이나 인후부 불편감은 없으며, 비흡연자이고 복용 중인 약물도 없다. 신체검사에서 혈압 120/80mmHg, 맥박 78회/분, 호흡 18회/분, 체온 36.5°C로 측정되었다. 청진상 호흡음은 정상이었고, 가슴 X선 사진과 코곁굴 X선 사진에서도 이상 소견이 없었다. 폐기능검사 결과는 다음과 같다.  
- 강제 폐활량(FVC): 정상 예측치의 91%  
- 1초간 강제날숨유량(FEV1): 정상 예측치의 85%  
- 1초간 강제날숨유량/강제폐활량(FEV1/FVC): 75%  

이 환자에서 다음으로 시행해야 할 검사는 무엇인가?  
1) 기관지내시경  
2) 기관지 확장제 반응 검사  
3) 가슴 컴퓨터단층촬영(CT) 
4) 메타콜린 기관지유발검사  
5) 폐 확산능 검사
4) 메타콜린 기관지유발


In [3]:
# 2) Tokenization (regex first, then Kiwi on remaining text)
KOREAN_RANGE = r"\uac00-\ud7a3"
BASE_STOPWORDS = set()


def normalize_text(text: str) -> str:
    return re.sub(r"\s+", " ", (text or "").lower()).strip()


def clean_token(token: str) -> str:
    token = normalize_text(token)
    # Keep Korean/English letters only. Remove digits and punctuation.
    return re.sub(r"^[^a-z\uac00-\ud7a3]+|[^a-z\uac00-\ud7a3]+$", "", token)


def load_stopwords(path: Path) -> set[str]:
    if not path.exists():
        return set()
    try:
        raw = json.loads(path.read_text(encoding='utf-8'))
    except Exception:
        return set()

    words = set()
    if isinstance(raw, list):
        words = {clean_token(str(x)) for x in raw if clean_token(str(x))}
    elif isinstance(raw, dict):
        for k, v in raw.items():
            ck = clean_token(str(k))
            if ck:
                words.add(ck)
            if isinstance(v, list):
                for item in v:
                    ci = clean_token(str(item))
                    if ci:
                        words.add(ci)
    return words


extra_stopwords_ko = load_stopwords(KOREAN_STOPWORDS_PATH)
extra_stopwords_en = load_stopwords(ENGLISH_STOPWORDS_PATH)
STOPWORDS = {clean_token(w) for w in BASE_STOPWORDS if clean_token(w)} | extra_stopwords_ko | extra_stopwords_en
print("stopwords =", len(STOPWORDS), f"(ko={len(extra_stopwords_ko)}, en={len(extra_stopwords_en)})")


MEDICAL_PATTERNS = [
    re.compile(r"\b[A-Z]{2,}[A-Z+\-_/]*\b"),
    re.compile(r"\b[a-z]{2,}[a-z+\-_/]*\b"),
    re.compile(rf"[{KOREAN_RANGE}]{{2,}}"),
]


try:
    from kiwipiepy import Kiwi
    kiwi = Kiwi()
    use_kiwi = True
except Exception as e:
    kiwi = None
    use_kiwi = False
    print("[warn] kiwipiepy not available. regex-only tokens will be used:", e)


def is_candidate(token: str) -> bool:
    if not token or token in STOPWORDS or len(token) <= 1:
        return False
    # Remove numeric tokens and alphanumeric tokens containing any digit.
    if re.search(r"\d", token):
        return False
    return bool(re.fullmatch(rf"[{KOREAN_RANGE}]{{2,}}|[a-z][a-z+\-_/]{{1,}}", token))


def extract_regex_keywords_and_remaining(text: str):
    matches = []
    for p in MEDICAL_PATTERNS:
        for m in p.finditer(text):
            token = clean_token(m.group(0))
            if is_candidate(token):
                matches.append((m.start(), m.end(), token))

    matches.sort(key=lambda x: (x[0], -(x[1] - x[0])))
    selected = []
    last_end = -1
    for s, e, t in matches:
        if s < last_end:
            continue
        selected.append((s, e, t))
        last_end = e

    keywords = [t for _, _, t in selected]

    remain_parts = []
    cur = 0
    for s, e, _ in selected:
        if cur < s:
            remain_parts.append(text[cur:s])
        cur = e
    if cur < len(text):
        remain_parts.append(text[cur:])

    remaining = re.sub(r"\s+", " ", " ".join(remain_parts)).strip()
    return keywords, remaining


def tokenize_doc(text: str):
    regex_tokens, remaining = extract_regex_keywords_and_remaining(text)

    kiwi_tokens = []
    if use_kiwi and remaining:
        for tok in kiwi.tokenize(remaining):
            term = clean_token(tok.form)
            if is_candidate(term):
                kiwi_tokens.append(term)

    return regex_tokens + kiwi_tokens


tokenized_docs = [tokenize_doc(doc) for doc in documents]
print("tokenized_docs =", len(tokenized_docs))
print("sample tokens =", tokenized_docs[0][:30] if tokenized_docs else [])


stopwords = 721 (ko=594, en=127)
tokenized_docs = 14994
sample tokens = ['여자가', '개월', '전부터', '기침을', '한다며', '내원했다', '기침은', '밤에', '누워', '자려고', '심해진다고', '한다', '전에도', '같은', '시기에', '기침이', '개월', '지속되다가', '저절로', '호전된', '병력이', '콧물이나', '인후부', '불편감은', '없으며', '비흡연자이고', '복용', '중인', '약물도', '없다']


In [8]:
# 3) Build TF/IDF and ko->en dictionary mapping
tf_counter = Counter()
df_counter = Counter()

for toks in tokenized_docs:
    tf_counter.update(toks)
    df_counter.update(set(toks))

total_terms = sum(tf_counter.values()) or 1
n_docs = len(tokenized_docs) or 1

rows = []
for token, count in tf_counter.items():
    token_doc_count = df_counter[token]  # 이 토큰이 등장한 문서 수
    tf = count / total_terms
    idf = math.log((1 + n_docs) / (1 + token_doc_count)) + 1.0
    tfidf = tf * idf
    rows.append({
        'keyword': token,
        'tf_count': int(count),
        'tf': float(tf),
        'doc_count': int(token_doc_count),  # 여기 수정
        'idf': float(idf),
        'tfidf': float(tfidf),
    })

keyword_stats_df = pd.DataFrame(rows).sort_values(['tfidf', 'tf_count'], ascending=[False, False]).reset_index(drop=True)

In [9]:
keyword_stats_df.to_csv(OUT_DIR / 'keyword_tfidf_scores_v2.csv', encoding='utf-8-sig')

In [ ]:
def is_korean_token(token: str) -> bool:
    token = token or ''
    return any(0xAC00 <= ord(ch) <= 0xD7A3 for ch in token)


def is_english_token(token: str) -> bool:
    return bool(re.search(r'[a-z]', token or ''))

# (A) Document-level Korean-English co-occurrence counts
pair_counter = Counter()
ko_doc_counter = Counter()

for toks in tokenized_docs:
    unique_tokens = set(toks)
    ko_tokens = {t for t in unique_tokens if is_korean_token(t)}
    en_tokens = {t for t in unique_tokens if is_english_token(t)}

    for ko in ko_tokens:
        ko_doc_counter[ko] += 1

    for ko in ko_tokens:
        for en in en_tokens:
            pair_counter[(ko, en)] += 1


# (B) Merge with manual dictionary if exists
manual_dict_path = RAW_DIR / 'ko_en_dictionary.json'
manual_dict = {}
if manual_dict_path.exists():
    try:
        loaded = json.loads(manual_dict_path.read_text(encoding='utf-8'))
        if isinstance(loaded, dict):
            for k, vals in loaded.items():
                ko = clean_token(str(k))
                if not ko:
                    continue
                aliases = []
                if isinstance(vals, list):
                    aliases = [clean_token(str(v)) for v in vals if clean_token(str(v))]
                manual_dict[ko] = sorted(set(aliases))
    except Exception as e:
        print('[warn] manual dictionary load failed:', e)

In [ ]:
# (C) Auto-generate ko->en mapping
TOP_N = 5000
MAX_ALIAS_PER_KO = 10
MIN_PAIR_COUNT = 2
MIN_PAIR_RATIO = 0.03

top_korean_keywords = [
    kw for kw in keyword_stats_df['keyword'].tolist()
    if is_korean_token(kw)
][:TOP_N]

ko_en_dictionary_auto = {} 
for ko in top_korean_keywords:
    candidates = []
    ko_docs = ko_doc_counter.get(ko, 0)

    for (ko_key, en), pair_cnt in pair_counter.items():
        if ko_key != ko:
            continue
        if pair_cnt < MIN_PAIR_COUNT:
            continue

        ratio = (pair_cnt / ko_docs) if ko_docs else 0.0
        if ratio < MIN_PAIR_RATIO:
            continue

        candidates.append((en, pair_cnt, ratio))

    candidates.sort(key=lambda x: (-x[1], -x[2], x[0]))
    auto_aliases = [en for en, _, _ in candidates[:MAX_ALIAS_PER_KO]]

    merged_aliases = set(auto_aliases)
    merged_aliases.update(manual_dict.get(ko, []))

    if merged_aliases:
        ko_en_dictionary_auto[ko] = sorted(merged_aliases)

print('keyword rows =', len(keyword_stats_df))
print('korean keywords considered =', len(top_korean_keywords))
print('dictionary size (non-empty aliases) =', len(ko_en_dictionary_auto))

sample_items = list(ko_en_dictionary_auto.items())[:20]
pd.DataFrame(sample_items, columns=['ko_keyword', 'en_aliases'])


In [ ]:
# 4) 결과 저장
keyword_stats_df.to_csv(OUTPUT_TFIDF_CSV, index=False, encoding='utf-8-sig')
with OUTPUT_DICT_JSON.open('w', encoding='utf-8') as f:
    json.dump(ko_en_dictionary_auto, f, ensure_ascii=False, indent=2)

print('saved tf-idf:', OUTPUT_TFIDF_CSV)
print('saved dictionary:', OUTPUT_DICT_JSON)
